# 할인 유입 고객의 12월 갱신·연간권 전환 분석

개인 프로젝트. `추가분석`~`추가분석4`의 핵심 분석과 검토 내용을 공개용으로 정리했습니다.

**분석 질문:** 할인 유입 고객은 갱신과 연간권 전환에 따라 어떤 학습 행동 차이가 있을까요?

11월 할인 유입 **797명 → 12월 갱신 368명 → 연간권 기록 136명**. 갱신자 중 **37.0%**입니다.

고객 원본·사용자별 출력은 포함하지 않습니다. 입력 CSV가 별도로 필요하며 실행 방법은 README를 확인하세요.

## 1. 입력 데이터와 점검

제공된 15,920원 플랜 고객 마스터, 갱신 로그, 콘텐츠 시작·완료 로그를 사용합니다. `SUBSCRIPTION_DATA_DIR` 또는 `data/` 폴더를 입력으로 받습니다.

In [1]:
from analysis import (load_tables, build_cohort, build_features, summarize,
                      save_charts, export_aggregates)
tables, input_checks = load_tables()
display(input_checks)

,table,rows,missing_user_id_rows,unique_users
0,master,4846,0,4846
1,renew,12776,0,6262
2,start,107945,0,37884
3,end,72802,0,19057


## 2. 할인 고객만으로 대상 통일

마스터에서 첫 결제월 2022-11, 첫 플랜 가격 15,920원, 할인 여부 1인 고객을 선택합니다. 갱신은 12월에 한정해 별도 판정합니다. 두 상품 기록이 모두 있으면 연간권 기록 그룹에 한 번만 포함합니다.

In [2]:
cohort = build_cohort(tables)
features = build_features(tables, cohort)
summary = summarize(tables, cohort, features)
display(summary['cohort_reconciliation'])

,cohort,users,dec_renew_users,dec_annual_users
0,"11월 전체 15,920원 플랜",886,429,144
1,11월 할인 고객,797,368,136


`886 → 429 → 144`는 정상가를 포함한 전체 11월 고객의 집계입니다. 할인 고객만의 흐름과 연결하지 않습니다. 다음 표의 `pct`는 각 행에 표시한 `denominator`를 분모로 계산합니다.

In [3]:
display(summary['funnel'].round(4))
print(f"할인 유입 전체 중 연간권 기록 비중: {summary['annual_share_of_all_discount_pct']:.4f}%")

,stage,users,denominator,pct
0,11월 할인 유입,797,797,100.0000
1,12월 갱신 기록,368,797,46.1731
2,연간권 기록,136,368,36.9565


할인 유입 전체 중 연간권 기록 비중: 17.0640%


![할인 코호트 퍼널](assets/discount_cohort_funnel.png)

## 3. 고객을 한 번씩 분류한 행동 비교

12월 갱신 기록 없음 / 월간권 기록만 / 연간권 기록 있음으로 배타적으로 분류합니다. ‘기록 없음’을 실제 탈퇴나 영구 이탈로 해석하지 않습니다.

콘텐츠 지표는 `[첫 결제, 첫 결제+30일)` 범위로 재계산합니다. 완주율은 같은 구간에서 시작한 고유 콘텐츠 중 완료 기록도 있는 콘텐츠의 비율입니다. 시작 콘텐츠가 없는 고객의 비율은 미정의로 두며 평균의 대상 고객 수도 함께 표시합니다.

활동 날짜 수와 최근 활동 간격은 원래 고객 마스터의 30일 전처리 값을 활용합니다.

In [4]:
display(summary['groups'].round(4))

,users,mean_active_days,mean_days_since_last_activity,mean_start_content_unique_30,mean_difficulty_count_30,mean_completion_rate_30_pct,completion_rate_defined_users,users_with_start_content_30
group,,,,,,,,
12월 갱신 기록 없음,429,6.2984,12.4662,3.7506,1.5524,32.1303,368,368
월간권 기록만,232,6.4828,10.0690,3.4224,1.5129,34.8300,192,192
연간권 기록 있음,136,11.2500,5.8897,5.1544,1.8971,46.5398,126,126


![학습 행동 비교](assets/learning_behavior_comparison.png)

## 4. 기존 요약과 비교 기준 구분

아래 값은 제공 고객 마스터 전체의 할인 여부별 원본 갱신 플래그 요약입니다. 여러 달 고객을 합친 지표로, 위 11월 코호트의 12월 갱신율과 구분합니다.

In [5]:
display(summary['master_rates'].round(4))

,users,renewed_in_source_master,source_master_renewal_pct
has_discount,,,
0,3547,2240,63.1520
1,1299,690,53.1178


## 5. 모델과 시간 순서 검토

원본 `추가분석4`는 Random Forest와 Extra Trees로 마스터의 `renewed`를 분류하고 변수 중요도를 확인했습니다. 여기의 `renewed_dec`와 타깃 정의가 다릅니다.

전체 기간으로 집계했던 난이도 수를 30일로 제한하면 155명의 값이 달라집니다. 12월 갱신자 중 일부는 30일 행동 관찰이 끝나기 전에 갱신했습니다. 따라서 이 행동 비교를 미래 갱신 예측 성능으로 읽을 수 없습니다. 원본 모형 점수를 새 타깃의 검증 성과로 옮기지 않았습니다.

[모델 검토 상세](model_review.md)에는 타깃·고객 중복·시간 범위·저장 출력의 차이와 이후 검증 방향을 정리했습니다.

In [6]:
display(summary['audit'])

{
  "cohort_unique_users": 797,
  "customers_with_both_dec_plans": 13,
  "source_master_renewed_in_discount_cohort": 408,
  "source_master_flag_vs_dec_target_disagreements": 42,
  "renewed_before_30d_observation_end": 91,
  "changed_difficulty_users_after_window_filter": 155,
  "master_first_activity_before_payment": 0,
  "master_last_activity_after_reference": 0,
  "master_active_days_max": 31,
  "completion_undefined_users": 111
}

## 6. 해석과 활용 제안

고객군별 학습 활동·콘텐츠 이용의 차이를 확인하고, 활동이 이어지는 고객에게 연간권 안내를 제공하는 가설을 제안합니다. 행동 차이나 변수 중요도는 인과효과가 아닙니다.

실제 CRM에 활용하려면 안내 시점까지의 행동만 사용하고, 무작위 비교로 연간권 전환·비용·후속 유지율을 확인해야 합니다. 업셀링·갱신 개선은 실제 실행 성과로 주장하지 않습니다.

[지표 정의](metric_notes.md) · [변경 내용](changes.md)

## 7. 집계와 그래프 저장

사용자별 레코드 대신 집계 JSON과 그래프만 저장합니다.

In [7]:
save_charts(summary)
_ = export_aggregates(summary)
print('집계 결과와 그래프 저장 완료')

집계 결과와 그래프 저장 완료
